# 광고 삽입 타이밍 최적화 — OpenCV + librosa + Silero VAD

**목표:** 움직임(모션)이 적고 소리(음성)가 없는 구간을 탐지하여 최적 광고 타이밍을 자동으로 결정

| 순서 | 대상 | 핵심 파라미터 | 기준 영상 |
|------|------|-------------|----------|
| 1 | OpenCV | threshold, frame_interval | SampleVideo.mp4 |
| 2 | librosa | top_db, min_silence_s | SampleVideo.mp4 |
| 3 | Silero VAD | threshold, min_silence_ms | SampleVideo.mp4 |
| 4 | librosa vs Silero VAD | 최적값 기준 비교 → 모델 선정 | SampleVideo.mp4 |
| 5 | 선정 모델 + OpenCV 검증 | 확정 최적값 고정 | 자취방 + 요리 |
| 6 | 최종 결합 | OpenCV + 침묵 탐지 | 3개 영상 전체 |

**실행 전 체크리스트:**
- [ ] 런타임 > 런타임 유형 변경 > **GPU (T4)** 선택
- [ ] Google Drive에 영상 업로드 확인
  - `내 드라이브/SampleVideo.Test/SampleVideo.mp4`
  - `내 드라이브/SampleVideo.Test/자취방.mp4` (자취방 원본)
  - `내 드라이브/SampleVideo.Test/요리.mp4` (요리 원본)

## 0. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# 패키지 설치
!pip install -q librosa soundfile tqdm
!pip install -q silero-vad
print("설치 완료!")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import librosa
import soundfile as sf
import subprocess
import pandas as pd
from tqdm.auto import tqdm
import torch

# ── 영상 경로 설정
BASE = "/content/drive/MyDrive/SampleVideo.Test"

VIDEOS = {
    "SampleVideo": os.path.join(BASE, "SampleVideo.mp4"),   # 튜닝용
    "자취방":       os.path.join(BASE, "자취방.mp4"),          # 검증용
    "요리":         os.path.join(BASE, "요리.mp4"),            # 검증용
}

# 경로 확인
for name, path in VIDEOS.items():
    exists = "OK" if os.path.exists(path) else "파일 없음"
    print(f"[{name}] {exists} — {path}")

## 공통 유틸 — 오디오 추출

In [ ]:
def extract_audio(video_path, out_wav="/content/audio_tmp.wav"):
    """영상에서 오디오(wav)를 추출"""
    cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-ac", "1",          # 모노
        "-ar", "16000",      # 16kHz (Silero VAD 요구사항)
        "-vn", out_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"오디오 추출 완료: {out_wav}")
    return out_wav

# SampleVideo 오디오 추출
audio_path = extract_audio(VIDEOS["SampleVideo"])
y, sr = librosa.load(audio_path, sr=16000)
duration = librosa.get_duration(y=y, sr=sr)
print(f"샘플레이트: {sr}Hz | 길이: {duration:.1f}초")

---
## Step 1. OpenCV 모션 탐지 파라미터 튜닝

**튜닝 파라미터:**
- `threshold` (10~60): 낮을수록 작은 움직임도 탐지
- `frame_interval` (1~30): 몇 프레임마다 비교할지

In [ ]:
def compute_motion_timeline(video_path, threshold=25, frame_interval=5):
    """OpenCV 프레임 차이 기반 모션 타임라인 계산"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    motion_scores = []  # (time_sec, motion_score)
    prev_gray = None
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            gray = cv2.GaussianBlur(gray, (5, 5), 0)  # 노이즈 제거
            if prev_gray is not None:
                diff = cv2.absdiff(prev_gray, gray)
                _, thresh = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
                score = cv2.countNonZero(thresh)
                time_sec = frame_idx / fps
                motion_scores.append({"time_sec": round(time_sec, 2), "motion_score": score})
            prev_gray = gray
        frame_idx += 1

    cap.release()
    return pd.DataFrame(motion_scores), fps

# ── 튜닝 실험
THRESHOLD_LIST    = [10, 20, 25, 30, 40, 50, 60]
FRAME_INTERVAL_LIST = [1, 3, 5, 10, 15, 30]
LOW_MOTION_RATIO  = 0.3  # 하위 30%를 저모션 구간으로 정의

opencv_results = []

print("[Step 1] OpenCV 모션 탐지 튜닝 시작")
print(f"threshold: {THRESHOLD_LIST}")
print(f"frame_interval: {FRAME_INTERVAL_LIST}")
print()

for thr in tqdm(THRESHOLD_LIST, desc="threshold"):
    for fi in FRAME_INTERVAL_LIST:
        df_motion, fps = compute_motion_timeline(VIDEOS["SampleVideo"], threshold=thr, frame_interval=fi)
        cutoff = df_motion["motion_score"].quantile(LOW_MOTION_RATIO)
        low_motion = df_motion[df_motion["motion_score"] <= cutoff]
        opencv_results.append({
            "threshold": thr,
            "frame_interval": fi,
            "총_프레임_수": len(df_motion),
            "저모션_구간_수": len(low_motion),
            "평균_모션_점수": round(df_motion["motion_score"].mean(), 1),
            "저모션_평균_점수": round(low_motion["motion_score"].mean(), 1),
        })

df_opencv = pd.DataFrame(opencv_results)
print("\n[OpenCV 튜닝 결과]")
print(df_opencv.to_string(index=False))

In [ ]:
# OpenCV 결과 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Step 1. OpenCV Motion Detection Tuning', fontsize=13, fontweight='bold')

# threshold별 저모션 구간 수
ax = axes[0]
for fi in FRAME_INTERVAL_LIST:
    sub = df_opencv[df_opencv["frame_interval"] == fi]
    ax.plot(sub["threshold"], sub["저모션_구간_수"], marker='o', label=f'interval={fi}')
ax.set_xlabel('threshold')
ax.set_ylabel('Low Motion Segments')
ax.set_title('threshold vs Low Motion Segments')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# frame_interval별 평균 모션 점수
ax = axes[1]
for thr in THRESHOLD_LIST:
    sub = df_opencv[df_opencv["threshold"] == thr]
    ax.plot(sub["frame_interval"], sub["평균_모션_점수"], marker='s', label=f'thr={thr}')
ax.set_xlabel('frame_interval')
ax.set_ylabel('Avg Motion Score')
ax.set_title('frame_interval vs Avg Motion Score')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('opencv_tuning_result.png', dpi=150, bbox_inches='tight')
plt.show()

# 최적값 선정 — 저모션 구간이 가장 안정적인 지점
best_opencv = df_opencv.sort_values("저모션_구간_수", ascending=False).iloc[0]
print(f"\n[OpenCV 추천 최적값]")
print(f"  threshold     = {best_opencv['threshold']}")
print(f"  frame_interval = {best_opencv['frame_interval']}")
print(f"  저모션 구간 수 = {best_opencv['저모션_구간_수']}")

BEST_OPENCV_THRESHOLD = int(best_opencv['threshold'])
BEST_OPENCV_INTERVAL  = int(best_opencv['frame_interval'])

---
## Step 2. librosa 침묵 탐지 파라미터 튜닝

**튜닝 파라미터:**
- `top_db` (20~60): 낮을수록 엄격하게 침묵 판단
- `min_silence_s` (0.5~3.0): 최소 침묵 길이

In [ ]:
def detect_silence_librosa(audio_path, top_db=30, min_silence_s=1.0):
    """librosa로 침묵 구간 탐지"""
    y, sr = librosa.load(audio_path, sr=16000)
    # 비침묵 구간 탐지 후 역으로 침묵 구간 계산
    non_silent = librosa.effects.split(y, top_db=top_db,
                                        frame_length=2048, hop_length=512)
    total_duration = librosa.get_duration(y=y, sr=sr)
    min_silence_samples = int(min_silence_s * sr)

    # 침묵 구간 계산 (비침묵 구간 사이사이)
    silence_intervals = []
    prev_end = 0
    for start, end in non_silent:
        if start - prev_end >= min_silence_samples:
            s_sec = round(prev_end / sr, 2)
            e_sec = round(start / sr, 2)
            silence_intervals.append((s_sec, e_sec, round(e_sec - s_sec, 2)))
        prev_end = end
    # 마지막 구간
    end_samples = len(y)
    if end_samples - prev_end >= min_silence_samples:
        s_sec = round(prev_end / sr, 2)
        e_sec = round(end_samples / sr, 2)
        silence_intervals.append((s_sec, e_sec, round(e_sec - s_sec, 2)))

    total_silence = sum(d for _, _, d in silence_intervals)
    return {
        "침묵_구간_수": len(silence_intervals),
        "총_침묵_시간(s)": round(total_silence, 1),
        "평균_침묵_길이(s)": round(total_silence / len(silence_intervals), 2) if silence_intervals else 0,
        "침묵_비율(%)": round(total_silence / total_duration * 100, 1),
        "intervals": silence_intervals
    }

# ── 튜닝 실험
TOP_DB_LIST      = [20, 25, 30, 35, 40, 50, 60]
MIN_SILENCE_LIST = [0.5, 1.0, 1.5, 2.0, 3.0]

librosa_results = []

print("[Step 2] librosa 침묵 탐지 튜닝 시작")
for top_db in tqdm(TOP_DB_LIST, desc="top_db"):
    for min_s in MIN_SILENCE_LIST:
        res = detect_silence_librosa(audio_path, top_db=top_db, min_silence_s=min_s)
        librosa_results.append({
            "top_db": top_db,
            "min_silence_s": min_s,
            "침묵_구간_수": res["침묵_구간_수"],
            "총_침묵_시간(s)": res["총_침묵_시간(s)"],
            "평균_침묵_길이(s)": res["평균_침묵_길이(s)"],
            "침묵_비율(%)": res["침묵_비율(%)"],
        })

df_librosa = pd.DataFrame(librosa_results)
print("\n[librosa 튜닝 결과]")
print(df_librosa.to_string(index=False))

In [ ]:
# librosa 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Step 2. librosa Silence Detection Tuning', fontsize=13, fontweight='bold')

ax = axes[0]
for min_s in MIN_SILENCE_LIST:
    sub = df_librosa[df_librosa["min_silence_s"] == min_s]
    ax.plot(sub["top_db"], sub["침묵_구간_수"], marker='o', label=f'min_s={min_s}s')
ax.set_xlabel('top_db')
ax.set_ylabel('Silence Segments')
ax.set_title('top_db vs Silence Segments')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
for min_s in MIN_SILENCE_LIST:
    sub = df_librosa[df_librosa["min_silence_s"] == min_s]
    ax.plot(sub["top_db"], sub["침묵_비율(%)"], marker='s', label=f'min_s={min_s}s')
ax.set_xlabel('top_db')
ax.set_ylabel('Silence Ratio (%)')
ax.set_title('top_db vs Silence Ratio')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('librosa_tuning_result.png', dpi=150, bbox_inches='tight')
plt.show()

# 추천 최적값 — 침묵 비율 20~40% 범위에서 구간 수가 많은 설정
candidates = df_librosa[(df_librosa["침묵_비율(%)"] >= 20) & (df_librosa["침묵_비율(%)"] <= 40)]
best_librosa = candidates.sort_values("침묵_구간_수", ascending=False).iloc[0]
print(f"\n[librosa 추천 최적값]")
print(f"  top_db        = {best_librosa['top_db']}")
print(f"  min_silence_s = {best_librosa['min_silence_s']}")
print(f"  침묵 구간 수   = {best_librosa['침묵_구간_수']}")
print(f"  침묵 비율      = {best_librosa['침묵_비율(%)']}%")

BEST_LIBROSA_TOP_DB   = best_librosa['top_db']
BEST_LIBROSA_MIN_S    = best_librosa['min_silence_s']

---
## Step 3. Silero VAD 파라미터 튜닝

**튜닝 파라미터:**
- `threshold` (0.3~0.7): 음성 판단 확률 임계값
- `min_silence_duration_ms` (300~2000ms): 최소 침묵 길이

In [ ]:
# Silero VAD 모델 로드
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vad_model = load_silero_vad()
vad_model = vad_model.to(device)
print(f"Silero VAD 로드 완료 | device: {device}")

wav_tensor = read_audio(audio_path)  # 16kHz mono tensor
wav_tensor = wav_tensor.to(device)
print(f"오디오 길이: {wav_tensor.shape[0] / 16000:.1f}초")

In [ ]:
def detect_silence_silero(wav_tensor, threshold=0.5, min_silence_ms=500):
    """Silero VAD로 음성 구간 탐지 후 침묵 구간 역산출"""
    SR = 16000
    total_duration = wav_tensor.shape[0] / SR

    speech_ts = get_speech_timestamps(
        wav_tensor, vad_model,
        threshold=threshold,
        min_silence_duration_ms=min_silence_ms,
        min_speech_duration_ms=100,
        return_seconds=True
    )

    # 침묵 구간 = 음성 구간 사이
    silence_intervals = []
    prev_end = 0.0
    for seg in speech_ts:
        if seg['start'] - prev_end >= (min_silence_ms / 1000):
            dur = round(seg['start'] - prev_end, 2)
            silence_intervals.append((round(prev_end, 2), round(seg['start'], 2), dur))
        prev_end = seg['end']
    if total_duration - prev_end >= (min_silence_ms / 1000):
        dur = round(total_duration - prev_end, 2)
        silence_intervals.append((round(prev_end, 2), round(total_duration, 2), dur))

    total_silence = sum(d for _, _, d in silence_intervals)
    return {
        "침묵_구간_수": len(silence_intervals),
        "총_침묵_시간(s)": round(total_silence, 1),
        "평균_침묵_길이(s)": round(total_silence / len(silence_intervals), 2) if silence_intervals else 0,
        "침묵_비율(%)": round(total_silence / total_duration * 100, 1),
        "intervals": silence_intervals
    }

# ── 튜닝 실험
THRESHOLD_LIST_VAD   = [0.3, 0.4, 0.5, 0.6, 0.7]
MIN_SILENCE_MS_LIST  = [300, 500, 700, 1000, 1500, 2000]

silero_results = []

print("[Step 3] Silero VAD 튜닝 시작")
for thr in tqdm(THRESHOLD_LIST_VAD, desc="threshold"):
    for ms in MIN_SILENCE_MS_LIST:
        res = detect_silence_silero(wav_tensor, threshold=thr, min_silence_ms=ms)
        silero_results.append({
            "threshold": thr,
            "min_silence_ms": ms,
            "침묵_구간_수": res["침묵_구간_수"],
            "총_침묵_시간(s)": res["총_침묵_시간(s)"],
            "평균_침묵_길이(s)": res["평균_침묵_길이(s)"],
            "침묵_비율(%)": res["침묵_비율(%)"],
        })

df_silero = pd.DataFrame(silero_results)
print("\n[Silero VAD 튜닝 결과]")
print(df_silero.to_string(index=False))

In [ ]:
# Silero VAD 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Step 3. Silero VAD Tuning', fontsize=13, fontweight='bold')

ax = axes[0]
for ms in MIN_SILENCE_MS_LIST:
    sub = df_silero[df_silero["min_silence_ms"] == ms]
    ax.plot(sub["threshold"], sub["침묵_구간_수"], marker='o', label=f'{ms}ms')
ax.set_xlabel('threshold')
ax.set_ylabel('Silence Segments')
ax.set_title('threshold vs Silence Segments')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
for ms in MIN_SILENCE_MS_LIST:
    sub = df_silero[df_silero["min_silence_ms"] == ms]
    ax.plot(sub["threshold"], sub["침묵_비율(%)"], marker='s', label=f'{ms}ms')
ax.set_xlabel('threshold')
ax.set_ylabel('Silence Ratio (%)')
ax.set_title('threshold vs Silence Ratio')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('silero_tuning_result.png', dpi=150, bbox_inches='tight')
plt.show()

candidates = df_silero[(df_silero["침묵_비율(%)"] >= 20) & (df_silero["침묵_비율(%)"] <= 40)]
best_silero = candidates.sort_values("침묵_구간_수", ascending=False).iloc[0]
print(f"\n[Silero VAD 추천 최적값]")
print(f"  threshold       = {best_silero['threshold']}")
print(f"  min_silence_ms  = {best_silero['min_silence_ms']}")
print(f"  침묵 구간 수     = {best_silero['침묵_구간_수']}")
print(f"  침묵 비율        = {best_silero['침묵_비율(%)']}%")

BEST_SILERO_THRESHOLD = best_silero['threshold']
BEST_SILERO_MIN_MS    = best_silero['min_silence_ms']

---
## Step 4. librosa vs Silero VAD 비교 → 최종 모델 선정

In [ ]:
# 두 모델 최적값으로 동일 영상 비교
res_lib = detect_silence_librosa(audio_path,
                                  top_db=BEST_LIBROSA_TOP_DB,
                                  min_silence_s=BEST_LIBROSA_MIN_S)
res_sil = detect_silence_silero(wav_tensor,
                                  threshold=BEST_SILERO_THRESHOLD,
                                  min_silence_ms=BEST_SILERO_MIN_MS)

print("=" * 55)
print("  librosa vs Silero VAD 최적값 비교")
print("=" * 55)
print(f"{'항목':<20} {'librosa':>12} {'Silero VAD':>12}")
print("-" * 55)
print(f"{'침묵 구간 수':<20} {res_lib['침묵_구간_수']:>12} {res_sil['침묵_구간_수']:>12}")
print(f"{'총 침묵 시간(s)':<20} {res_lib['총_침묵_시간(s)']:>12} {res_sil['총_침묵_시간(s)']:>12}")
print(f"{'평균 침묵 길이(s)':<20} {res_lib['평균_침묵_길이(s)']:>12} {res_sil['평균_침묵_길이(s)']:>12}")
print(f"{'침묵 비율(%)':<20} {res_lib['침묵_비율(%)']:>12} {res_sil['침묵_비율(%)']:>12}")
print("=" * 55)

# 광고 삽입 후보 타이밍 (1초 이상 침묵만)
ad_candidates_lib = [(s, e, d) for s, e, d in res_lib['intervals'] if d >= 1.0]
ad_candidates_sil = [(s, e, d) for s, e, d in res_sil['intervals'] if d >= 1.0]
print(f"\n광고 후보 타이밍 (1초 이상 침묵):")
print(f"  librosa    : {len(ad_candidates_lib)}개")
print(f"  Silero VAD : {len(ad_candidates_sil)}개")

In [ ]:
# 파형 + 침묵 구간 시각화 비교
import librosa.display

fig, axes = plt.subplots(2, 1, figsize=(18, 8))
fig.suptitle('Step 4. librosa vs Silero VAD — Silence Detection Comparison', fontsize=13, fontweight='bold')

y_vis, sr_vis = librosa.load(audio_path, sr=16000)
times = np.linspace(0, len(y_vis) / sr_vis, len(y_vis))

for ax, title, intervals in [
    (axes[0], f'librosa (top_db={BEST_LIBROSA_TOP_DB}, min_s={BEST_LIBROSA_MIN_S}s)', res_lib['intervals']),
    (axes[1], f'Silero VAD (threshold={BEST_SILERO_THRESHOLD}, min_ms={BEST_SILERO_MIN_MS}ms)', res_sil['intervals'])
]:
    ax.plot(times, y_vis, color='steelblue', alpha=0.6, linewidth=0.5)
    for s, e, d in intervals:
        ax.axvspan(s, e, alpha=0.3, color='tomato', label='_' if s > 0 else '침묵 구간')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('silence_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장: silence_comparison.png")

In [ ]:
# 최종 모델 선정 — 아래에서 직접 선택
# Silero VAD가 음성 구분 정밀도가 높으므로 기본 추천
SELECTED_MODEL = "Silero VAD"  # 또는 "librosa"

if SELECTED_MODEL == "Silero VAD":
    BEST_SILENCE_INTERVALS = res_sil['intervals']
    BEST_SILENCE_PARAMS = {"threshold": BEST_SILERO_THRESHOLD, "min_silence_ms": BEST_SILERO_MIN_MS}
else:
    BEST_SILENCE_INTERVALS = res_lib['intervals']
    BEST_SILENCE_PARAMS = {"top_db": BEST_LIBROSA_TOP_DB, "min_silence_s": BEST_LIBROSA_MIN_S}

print(f"선정 모델: {SELECTED_MODEL}")
print(f"최적 파라미터: {BEST_SILENCE_PARAMS}")

---
## Step 5. 선정 모델 + OpenCV — 자취방 & 요리 영상 검증

In [ ]:
validation_results = []

for video_name in ["자취방", "요리"]:
    video_path = VIDEOS[video_name]
    print(f"\n[{video_name}] 검증 시작...")

    # 오디오 추출
    audio_tmp = f"/content/audio_{video_name}.wav"
    extract_audio(video_path, out_wav=audio_tmp)

    # OpenCV 모션 탐지
    df_motion, fps = compute_motion_timeline(video_path,
                                             threshold=BEST_OPENCV_THRESHOLD,
                                             frame_interval=BEST_OPENCV_INTERVAL)
    cutoff = df_motion["motion_score"].quantile(LOW_MOTION_RATIO)
    low_motion_times = set(df_motion[df_motion["motion_score"] <= cutoff]["time_sec"].tolist())

    # 침묵 탐지
    if SELECTED_MODEL == "Silero VAD":
        wav_val = read_audio(audio_tmp).to(device)
        res_silence = detect_silence_silero(wav_val, **BEST_SILENCE_PARAMS)
    else:
        res_silence = detect_silence_librosa(audio_tmp, **BEST_SILENCE_PARAMS)

    # 교집합 — 저모션 + 침묵 겹치는 구간
    ad_candidates = []
    for s, e, d in res_silence['intervals']:
        if d < 0.5:
            continue
        # 해당 침묵 구간 내 저모션 시간이 존재하는지 확인
        overlap = [t for t in low_motion_times if s <= t <= e]
        if overlap:
            ad_candidates.append({"start": s, "end": e, "duration": d})

    validation_results.append({
        "영상": video_name,
        "저모션_구간_수": len(low_motion_times),
        "침묵_구간_수": res_silence["침묵_구간_수"],
        "침묵_비율(%)": res_silence["침묵_비율(%)"],
        "광고_후보_타이밍_수": len(ad_candidates),
    })
    print(f"  저모션 구간: {len(low_motion_times)}개")
    print(f"  침묵 구간:   {res_silence['침묵_구간_수']}개 ({res_silence['침묵_비율(%)']}%)")
    print(f"  광고 후보:   {len(ad_candidates)}개")

df_val = pd.DataFrame(validation_results)
print("\n[검증 결과]")
print(df_val.to_string(index=False))

---
## Step 6. 최종 결합 — 3개 영상 광고 타이밍 출력

In [ ]:
final_results = []

for video_name, video_path in VIDEOS.items():
    print(f"\n[{video_name}] 최종 광고 타이밍 분석...")

    audio_tmp = f"/content/audio_final_{video_name}.wav"
    extract_audio(video_path, out_wav=audio_tmp)

    # OpenCV
    df_motion, fps = compute_motion_timeline(video_path,
                                             threshold=BEST_OPENCV_THRESHOLD,
                                             frame_interval=BEST_OPENCV_INTERVAL)
    cutoff = df_motion["motion_score"].quantile(LOW_MOTION_RATIO)
    low_motion_times = set(df_motion[df_motion["motion_score"] <= cutoff]["time_sec"].tolist())

    # 침묵 탐지
    if SELECTED_MODEL == "Silero VAD":
        wav_val = read_audio(audio_tmp).to(device)
        res_silence = detect_silence_silero(wav_val, **BEST_SILENCE_PARAMS)
    else:
        res_silence = detect_silence_librosa(audio_tmp, **BEST_SILENCE_PARAMS)

    # 최적 광고 타이밍
    ad_timings = []
    for s, e, d in res_silence['intervals']:
        if d < 0.5:
            continue
        overlap = [t for t in low_motion_times if s <= t <= e]
        if overlap:
            ad_timings.append({"start_sec": s, "end_sec": e, "duration_sec": d})

    final_results.append({
        "영상": video_name,
        "저모션_구간_수": len(low_motion_times),
        "침묵_구간_수": res_silence["침묵_구간_수"],
        "광고_타이밍_수": len(ad_timings),
        "광고_타이밍_목록": ad_timings[:5]  # 상위 5개만 미리보기
    })

    print(f"  ✅ 최적 광고 타이밍 {len(ad_timings)}개 탐지")
    for t in ad_timings[:5]:
        print(f"     {t['start_sec']}s ~ {t['end_sec']}s ({t['duration_sec']}s)")

# 최종 요약 출력
print("\n" + "=" * 55)
print("  최종 광고 타이밍 요약")
print("=" * 55)
for r in final_results:
    print(f"[{r['영상']:10s}] 광고 후보: {r['광고_타이밍_수']}개")
print("=" * 55)
print(f"\n최종 확정 파라미터:")
print(f"  OpenCV      threshold={BEST_OPENCV_THRESHOLD}, frame_interval={BEST_OPENCV_INTERVAL}")
print(f"  {SELECTED_MODEL:<12} {BEST_SILENCE_PARAMS}")

In [ ]:
# 결과 CSV 저장 & Drive 백업
import shutil

df_opencv.to_csv("opencv_tuning.csv", index=False)
df_librosa.to_csv("librosa_tuning.csv", index=False)
df_silero.to_csv("silero_tuning.csv", index=False)

SAVE_DIR = "/content/drive/MyDrive/SampleVideo.Test/ad_timing_results"
os.makedirs(SAVE_DIR, exist_ok=True)
for fname in ["opencv_tuning.csv", "librosa_tuning.csv", "silero_tuning.csv",
              "opencv_tuning_result.png", "librosa_tuning_result.png",
              "silero_tuning_result.png", "silence_comparison.png"]:
    if os.path.exists(fname):
        shutil.copy(fname, SAVE_DIR)

print(f"결과 저장 완료: {SAVE_DIR}")

# Report용 출력
print("\n" + "=" * 60)
print("  아래 내용을 Project_Report.md 섹션 15에 붙여넣으세요")
print("=" * 60)
print(f"\n#### Step 1. OpenCV 최적값")
print(f"- threshold = {BEST_OPENCV_THRESHOLD}")
print(f"- frame_interval = {BEST_OPENCV_INTERVAL}")
print(f"\n#### Step 2~3. 침묵 탐지 최적값")
print(f"- 선정 모델: {SELECTED_MODEL}")
print(f"- 파라미터: {BEST_SILENCE_PARAMS}")
print(f"\n#### Step 4. librosa vs Silero VAD 비교")
print(f"- librosa   침묵 구간: {res_lib['침묵_구간_수']}개 / 침묵 비율: {res_lib['침묵_비율(%)']}%")
print(f"- Silero VAD 침묵 구간: {res_sil['침묵_구간_수']}개 / 침묵 비율: {res_sil['침묵_비율(%)']}%")
print(f"- 선정: {SELECTED_MODEL}")